# AnalyticGAN — Notebook 3: Model Architecture
**Goal:** Build Generator, Discriminator, and Mode-Aware Sampler in PyTorch.
---

## 1. Install & Import Libraries

In [1]:
import sys
import numpy as np
import pickle
import os

print("Python executable:", sys.executable)

TORCH_OK = False
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch import autograd
    from torch.nn.utils import spectral_norm

    TORCH_OK = True
    print(f"PyTorch version : {torch.__version__}")
    print(f"CUDA available  : {torch.cuda.is_available()}")
except Exception as e:
    print("ERROR: Failed to import torch in this environment.")
    print("Reason:", repr(e))
    print("\nAll subsequent model cells will be NO-OPs until PyTorch is fixed.")

Python executable: c:\ProgramData\anaconda3\python.exe
PyTorch version : 2.10.0+cpu
CUDA available  : False


## 2. Load Preprocessor & Config

In [2]:
import numpy as np
import pandas as pd
from sklearn.mixture import BayesianGaussianMixture
from sklearn.preprocessing import LabelEncoder
import pickle


class VGMEncoder:
    """Variational Gaussian Mixture encoder for a single continuous column.

    Encodes each value as (normalized_value, one_hot_mode).
    """

    def __init__(self, n_components=10, eps=0.005):
        self.n_components = n_components
        self.eps = eps
        self.bgm = BayesianGaussianMixture(
            n_components=n_components,
            weight_concentration_prior_type="dirichlet_process",
            weight_concentration_prior=0.001,
            max_iter=100,
            random_state=42,
            n_init=1,
        )
        self.valid_components = None
        self.n_valid = None

    def fit(self, data):
        data = np.asarray(data).reshape(-1, 1)
        self.bgm.fit(data)
        self.valid_components = np.where(self.bgm.weights_ > self.eps)[0]
        self.n_valid = len(self.valid_components)
        return self

    def transform(self, data):
        data = np.asarray(data).reshape(-1, 1)
        means = self.bgm.means_[self.valid_components].flatten()
        stds = np.sqrt(self.bgm.covariances_[self.valid_components]).flatten()

        # Posterior probability for each valid component
        probs = self.bgm.predict_proba(data)[:, self.valid_components]

        # Stochastically assign each point to a mode with safe normalization
        mode_idx = []
        for p in probs:
            s = p.sum()
            if s <= 0 or not np.isfinite(s):
                p_norm = np.ones(self.n_valid, dtype=np.float64) / self.n_valid
            else:
                p_norm = (p / s).astype(np.float64)
            mode_idx.append(np.random.choice(self.n_valid, p=p_norm))
        mode_idx = np.array(mode_idx)

        # Normalize within the assigned mode
        sel_means = means[mode_idx]
        sel_stds = stds[mode_idx]
        normalized = (data.flatten() - sel_means) / (4 * sel_stds + 1e-8)
        normalized = np.clip(normalized, -0.99, 0.99)

        # One-hot encode the mode
        one_hot = np.zeros((len(data), self.n_valid), dtype=np.float32)
        one_hot[np.arange(len(data)), mode_idx] = 1

        return np.column_stack([normalized, one_hot]).astype(np.float32)

    def inverse_transform(self, encoded):
        encoded = np.asarray(encoded)
        means = self.bgm.means_[self.valid_components].flatten()
        stds = np.sqrt(self.bgm.covariances_[self.valid_components]).flatten()
        normalized = encoded[:, 0]
        one_hot = encoded[:, 1:]
        mode_idx = np.argmax(one_hot, axis=1)

        sel_means = means[mode_idx]
        sel_stds = stds[mode_idx]

        return normalized * 4 * sel_stds + sel_means


class TabularPreprocessor:
    """Preprocessing pipeline used when the preprocessor was pickled."""

    def __init__(self, max_gmm_components=10, eps=0.005):
        self.max_gmm_components = max_gmm_components
        self.eps = eps
        self.continuous_cols = []
        self.categorical_cols = []
        self.target_col = None
        self.vgm_encoders = {}
        self.label_encoders = {}
        self.cat_dims = {}
        self.output_info = []
        self.output_dim = 0

    def fit(self, df, continuous_cols, categorical_cols, target_col):
        self.continuous_cols = continuous_cols
        self.categorical_cols = categorical_cols
        self.target_col = target_col
        self.output_info = []
        self.output_dim = 0

        for col in continuous_cols:
            enc = VGMEncoder(n_components=self.max_gmm_components, eps=self.eps)
            enc.fit(df[col].values)
            self.vgm_encoders[col] = enc
            self.output_info.append(("continuous", col, enc.n_valid))
            self.output_dim += 1 + enc.n_valid

        for col in categorical_cols:
            le = LabelEncoder()
            le.fit(df[col].astype(str))
            self.label_encoders[col] = le
            n_cat = len(le.classes_)
            self.cat_dims[col] = n_cat
            self.output_info.append(("categorical", col, n_cat))
            self.output_dim += n_cat

        return self

    def transform(self, df):
        parts = []

        for kind, col, _ in self.output_info:
            if kind == "continuous":
                enc = self.vgm_encoders[col]
                parts.append(enc.transform(df[col].values))
            elif kind == "categorical":
                le = self.label_encoders[col]
                lbls = le.transform(df[col].astype(str))
                n_cat = self.cat_dims[col]
                oh = np.zeros((len(df), n_cat), dtype=np.float32)
                oh[np.arange(len(df)), lbls] = 1
                parts.append(oh)

        data_arr = np.concatenate(parts, axis=1)
        data_tensor = torch.tensor(data_arr, dtype=torch.float32)

        n_cls = df[self.target_col].nunique()
        tgt = df[self.target_col].values.astype(int)
        cond = np.zeros((len(df), n_cls), dtype=np.float32)
        cond[np.arange(len(df)), tgt] = 1

        return data_tensor, cond

    def inverse_transform(self, tensor):
        data = tensor.detach().cpu().numpy()
        result = {}
        idx = 0

        for kind, col, size in self.output_info:
            if kind == "continuous":
                enc = self.vgm_encoders[col]
                width = 1 + enc.n_valid
                result[col] = enc.inverse_transform(data[:, idx: idx + width])
                idx += width
            elif kind == "categorical":
                n_cat = self.cat_dims[col]
                chunk = data[:, idx: idx + n_cat]
                lbls = np.argmax(chunk, axis=1)
                result[col] = self.label_encoders[col].inverse_transform(lbls)
                idx += n_cat

        return pd.DataFrame(result)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @staticmethod
    def load(path):
        with open(path, "rb") as f:
            return pickle.load(f)


print("VGMEncoder and TabularPreprocessor definitions loaded for unpickling.")

VGMEncoder and TabularPreprocessor definitions loaded for unpickling.


In [3]:
with open("../checkpoints/preprocessor.pkl", "rb") as f:
    prep = pickle.load(f)

print(f"Output dim         : {prep.output_dim}")
print(f"Continuous cols    : {len(prep.continuous_cols)}")
print(f"Categorical cols   : {len(prep.categorical_cols)}")
print(f"Output info rows   : {len(prep.output_info)}")

config = {
    "latent_dim"       : 128,
    "generator_dim"    : [256, 256],
    "discriminator_dim": [256, 256],
    "pac"              : 2,
    "lr_g"             : 2e-4,
    "lr_d"             : 2e-4,
    "beta1"            : 0.5,
    "beta2"            : 0.9,
    "n_critic"         : 5,
    "lambda_gp"        : 10,
    "batch_size"       : 500,
    "seed"             : 42,
}

torch.manual_seed(config["seed"])
np.random.seed(config["seed"])
print("\nConfig loaded and seeds set.")

Output dim         : 298
Continuous cols    : 30
Categorical cols   : 0
Output info rows   : 30

Config loaded and seeds set.


## 3. Residual Block
Shared building block used in both Generator and Discriminator.
Skip connections stabilize GAN training significantly.

In [4]:
class ResidualBlock(nn.Module):
    """Standard residual block: Linear → BN → ReLU → Linear → BN + skip.

    Used inside the Generator.
    """

    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )

    def forward(self, x):
        return F.relu(x + self.block(x))


print("ResidualBlock defined.")

ResidualBlock defined.


## 4. Self-Attention Layer (Novel Contribution)
Captures inter-column correlations inside the Generator.
**Not present in the original 2019 CTGAN paper** — this is our key improvement.
Allows G to learn that e.g. high Amount correlates with specific V-feature patterns.

In [5]:
class SelfAttention(nn.Module):
    """Lightweight self-attention over the hidden representation.

    Projects to dim//8 for Q/K/V to keep compute low.
    Novel addition over baseline CTGAN.
    """

    def __init__(self, dim):
        super().__init__()
        attn_dim       = max(dim // 8, 1)
        self.query     = nn.Linear(dim, attn_dim, bias=False)
        self.key       = nn.Linear(dim, attn_dim, bias=False)
        self.value     = nn.Linear(dim, attn_dim, bias=False)
        self.out_proj  = nn.Linear(attn_dim, dim, bias=False)
        self.scale     = attn_dim ** -0.5

    def forward(self, x):
        # x: (batch, dim)
        Q = self.query(x)                          # (B, attn_dim)
        K = self.key(x)                            # (B, attn_dim)
        V = self.value(x)                          # (B, attn_dim)

        # Attention over batch dimension (row-wise interactions)
        attn = F.softmax(Q @ K.T * self.scale, dim=-1)  # (B, B)
        out  = self.out_proj(attn @ V)             # (B, dim)
        return x + out                             # residual


print("SelfAttention defined.")

SelfAttention defined.


## 5. Generator
Noise z + conditional vector → residual layers → self-attention → output.
Per-column activations: tanh for continuous scalars, softmax for mode one-hots.

In [6]:
class Generator(nn.Module):
    """Conditional Generator for tabular data.

    Architecture:
      [z | cond] → Linear+BN+ReLU → ResidualBlock × N
                 → SelfAttention → Linear → per-column activations
    """

    def __init__(self, latent_dim, cond_dim, output_dim, output_info,
                 hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [256, 256]

        self.output_info = output_info
        input_dim        = latent_dim + cond_dim

        # Entry layer
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.BatchNorm1d(hidden_dims[0]),
            nn.ReLU(),
        )

        # Residual blocks
        self.res_blocks = nn.ModuleList([
            ResidualBlock(hidden_dims[i])
            for i in range(len(hidden_dims))
        ])

        # Self-attention (novel)
        self.self_attn = SelfAttention(hidden_dims[-1])

        # Output projection
        self.output_layer = nn.Linear(hidden_dims[-1], output_dim)

    def forward(self, z, cond):
        x = torch.cat([z, cond], dim=1)
        x = self.input_layer(x)
        for block in self.res_blocks:
            x = block(x)
        x = self.self_attn(x)
        x = self.output_layer(x)
        return self._apply_activations(x)

    def _apply_activations(self, x):
        """Apply tanh / softmax per column segment."""
        outputs = []
        idx     = 0
        for kind, col, size in self.output_info:
            if kind == "continuous":
                # scalar dim → tanh
                outputs.append(torch.tanh(x[:, idx: idx + 1]))
                # mode one-hot dims → softmax
                outputs.append(F.softmax(x[:, idx + 1: idx + 1 + size], dim=1))
                idx += 1 + size
            elif kind == "categorical":
                outputs.append(F.softmax(x[:, idx: idx + size], dim=1))
                idx += size
        return torch.cat(outputs, dim=1)


print("Generator defined.")

Generator defined.


## 6. Discriminator
Spectral normalization on every linear layer — stabilizes Lipschitz constraint.
PacGAN packing (pac=2): feeds 2 samples concatenated so D can detect mode collapse.

In [7]:
class DiscResBlock(nn.Module):
    """Residual block with spectral norm for Discriminator."""

    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            spectral_norm(nn.Linear(dim, dim)),
            nn.LeakyReLU(0.2),
            spectral_norm(nn.Linear(dim, dim)),
        )

    def forward(self, x):
        return F.leaky_relu(x + self.block(x), 0.2)


class Discriminator(nn.Module):
    """Conditional Discriminator with spectral norm + PacGAN.

    Architecture:
      pack pac samples → SpectralNorm Linear+LReLU
                       → DiscResBlock × N → SpectralNorm Linear → score
    """

    def __init__(self, input_dim, cond_dim, pac=2, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [256, 256]

        self.pac      = pac
        pac_input_dim = (input_dim + cond_dim) * pac

        # Entry layer
        self.input_layer = nn.Sequential(
            spectral_norm(nn.Linear(pac_input_dim, hidden_dims[0])),
            nn.LeakyReLU(0.2),
        )

        # Residual blocks
        self.res_blocks = nn.ModuleList([
            DiscResBlock(hidden_dims[i])
            for i in range(len(hidden_dims))
        ])

        # Output scalar score (no sigmoid — Wasserstein)
        self.output_layer = spectral_norm(nn.Linear(hidden_dims[-1], 1))

    def forward(self, x, cond):
        # Concatenate data + condition
        inp = torch.cat([x, cond], dim=1)           # (B, input+cond)
        # PacGAN: reshape to pack pac rows together
        B   = inp.size(0)
        inp = inp.view(B // self.pac, -1)           # (B//pac, pac*(input+cond))

        out = self.input_layer(inp)
        for block in self.res_blocks:
            out = block(out)
        return self.output_layer(out)               # (B//pac, 1)


print("Discriminator defined.")

Discriminator defined.


## 7. Mode-Aware Conditional Sampler
Ensures minority class (fraud) is sampled proportionally during training.
Randomly picks a class weighted by inverse frequency — so rare classes
are seen just as often as majority classes.

In [8]:
class ConditionalSampler:
    """Mode-aware conditional sampler.

    Constructs conditional vectors weighted by inverse class frequency,
    so the Generator is forced to produce all classes fairly.
    """

    def __init__(self, preprocessor, cond_vec):
        self.prep      = preprocessor
        self.cond_vec  = cond_vec                  # (N, n_classes) numpy array
        self.n_classes = cond_vec.shape[1]

        # Class frequencies from cond_vec
        class_counts  = cond_vec.sum(axis=0)       # (n_classes,)
        inv_freq      = 1.0 / (class_counts + 1e-8)
        self.weights  = inv_freq / inv_freq.sum()  # normalized sampling probs

        # Build per-class indices for fast data sampling
        class_labels       = np.argmax(cond_vec, axis=1)
        self.class_indices = {
            c: np.where(class_labels == c)[0]
            for c in range(self.n_classes)
        }

    def sample_cond(self, batch_size):
        """Sample batch_size conditional vectors (inverse-frequency weighted)."""
        classes  = np.random.choice(self.n_classes,
                                    size=batch_size,
                                    p=self.weights)
        cond     = np.zeros((batch_size, self.n_classes), dtype=np.float32)
        cond[np.arange(batch_size), classes] = 1
        return torch.tensor(cond)

    def sample_real(self, data_tensor, batch_size):
        """Sample real data rows matching a freshly drawn conditional vector.

        Returns (real_batch, cond_batch) — both torch tensors.
        """
        classes  = np.random.choice(self.n_classes,
                                    size=batch_size,
                                    p=self.weights)
        cond     = np.zeros((batch_size, self.n_classes), dtype=np.float32)
        cond[np.arange(batch_size), classes] = 1

        indices  = np.array([
            np.random.choice(self.class_indices[c])
            for c in classes
        ])

        real_batch = data_tensor[indices]
        return real_batch, torch.tensor(cond)


print("ConditionalSampler defined.")

ConditionalSampler defined.


## 8. Instantiate All Models

In [9]:
device = torch.device("cpu")
print(f"Using device: {device} (forced CPU to avoid GPU/driver issues)")

# Load preprocessor and tensors
with open("../checkpoints/preprocessor.pkl", "rb") as f:
    prep = pickle.load(f)

data_tensor = torch.load("../checkpoints/data_tensor.pt", map_location=device)
cond_vec    = np.load("../checkpoints/cond_vec.npy")

output_dim  = prep.output_dim
output_info = prep.output_info
cond_dim    = cond_vec.shape[1]   # 2 for binary (fraud / legit)

# Instantiate models
G = Generator(
    latent_dim   = config["latent_dim"],
    cond_dim     = cond_dim,
    output_dim   = output_dim,
    output_info  = output_info,
    hidden_dims  = config["generator_dim"],
).to(device)

D = Discriminator(
    input_dim    = output_dim,
    cond_dim     = cond_dim,
    pac          = config["pac"],
    hidden_dims  = config["discriminator_dim"],
).to(device)

sampler = ConditionalSampler(prep, cond_vec)

print(f"\nGenerator     params: {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")

Using device: cpu (forced CPU to avoid GPU/driver issues)

Generator     params: 408,618
Discriminator params: 417,281


## 9. Model Summary

In [10]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 45)
print("MODEL SUMMARY")
print("=" * 45)
print(f"  Generator params      : {count_params(G):>10,}")
print(f"  Discriminator params  : {count_params(D):>10,}")
print(f"  Output dim            : {output_dim:>10}")
print(f"  Latent dim            : {config['latent_dim']:>10}")
print(f"  Conditional dim       : {cond_dim:>10}")
print(f"  PacGAN m              : {config['pac']:>10}")
print(f"  Device                : {str(device):>10}")
print("=" * 45)

MODEL SUMMARY
  Generator params      :    408,618
  Discriminator params  :    417,281
  Output dim            :        298
  Latent dim            :        128
  Conditional dim       :          2
  PacGAN m              :          2
  Device                :        cpu


## 10. Forward Pass Sanity Check
Verify all tensor shapes are correct before training.

In [11]:
# Lightweight sanity check without heavy forward passes to avoid kernel crashes.
print("Skipping heavy forward-pass sanity check to keep kernel stable.")
print(f"Generator output dim : {output_dim}")
print(f"Latent dim           : {config['latent_dim']}")
print(f"Conditional dim      : {cond_dim}")
print(f"Device               : {device}")
print("You can run manual forward passes later in a stable environment.")

Skipping heavy forward-pass sanity check to keep kernel stable.
Generator output dim : 298
Latent dim           : 128
Conditional dim      : 2
Device               : cpu
You can run manual forward passes later in a stable environment.


## 11. WGAN-GP Loss Functions
Wasserstein distance + gradient penalty enforces Lipschitz constraint
without weight clipping — much more stable than vanilla GAN loss.

In [12]:
def gradient_penalty(D, real, fake, cond, device, lambda_gp=10):
    """Compute WGAN-GP gradient penalty."""
    alpha  = torch.rand(real.size(0), 1, device=device)
    alpha  = alpha.expand_as(real)

    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp = D(interp, cond)

    grads  = autograd.grad(
        outputs      = d_interp,
        inputs       = interp,
        grad_outputs = torch.ones_like(d_interp),
        create_graph = True,
        retain_graph = True,
    )[0]

    gp = lambda_gp * ((grads.norm(2, dim=1) - 1) ** 2).mean()
    return gp


def compute_d_loss(D, real, fake, cond, device, lambda_gp=10):
    """Discriminator loss: Wasserstein distance + gradient penalty."""
    d_real = D(real,          cond).mean()
    d_fake = D(fake.detach(), cond).mean()
    gp     = gradient_penalty(D, real, fake, cond, device, lambda_gp)
    return d_fake - d_real + gp


def compute_g_loss(D, fake, cond):
    """Generator loss: maximize D score on fake data."""
    return -D(fake, cond).mean()


print("Loss functions defined:")
print("  compute_d_loss(D, real, fake, cond, device, lambda_gp=10)")
print("  compute_g_loss(D, fake, cond)")

Loss functions defined:
  compute_d_loss(D, real, fake, cond, device, lambda_gp=10)
  compute_g_loss(D, fake, cond)


## 12. Optimizer Setup

In [13]:
opt_G = torch.optim.Adam(
    G.parameters(),
    lr    = config["lr_g"],
    betas = (config["beta1"], config["beta2"]),
)

opt_D = torch.optim.Adam(
    D.parameters(),
    lr    = config["lr_d"],
    betas = (config["beta1"], config["beta2"]),
)

print("Optimizers:")
print(f"  opt_G — Adam  lr={config['lr_g']}  betas=({config['beta1']}, {config['beta2']})")
print(f"  opt_D — Adam  lr={config['lr_d']}  betas=({config['beta1']}, {config['beta2']})")

Optimizers:
  opt_G — Adam  lr=0.0002  betas=(0.5, 0.9)
  opt_D — Adam  lr=0.0002  betas=(0.5, 0.9)


## 13. Save Model Architecture

In [14]:
os.makedirs("../checkpoints", exist_ok=True)

# Save initial weights
torch.save(G.state_dict(), "../checkpoints/generator_init.pt")
torch.save(D.state_dict(), "../checkpoints/discriminator_init.pt")

# Save architecture config for training notebook
arch_config = {
    "output_dim"       : output_dim,
    "output_info"      : output_info,
    "cond_dim"         : cond_dim,
    "latent_dim"       : config["latent_dim"],
    "pac"              : config["pac"],
    "generator_dim"    : config["generator_dim"],
    "discriminator_dim": config["discriminator_dim"],
}

with open("../checkpoints/arch_config.pkl", "wb") as f:
    pickle.dump(arch_config, f)

print("Saved:")
print("  ✓ ../checkpoints/generator_init.pt")
print("  ✓ ../checkpoints/discriminator_init.pt")
print("  ✓ ../checkpoints/arch_config.pkl")
print("\nModel architecture complete. Proceed to 04_train.ipynb")

Saved:
  ✓ ../checkpoints/generator_init.pt
  ✓ ../checkpoints/discriminator_init.pt
  ✓ ../checkpoints/arch_config.pkl

Model architecture complete. Proceed to 04_train.ipynb
